# postpro

选择 z 位置 (步进器 ◀◀ ◀ ▶ ▶▶, 对应原 postpro 的步进功能) -> 统计表 +
相空间图自动刷新; 下方合并了原 07 的切片/BFF/导出。发射度单位
"π mm mrad" 与 ASTRA 打印一致。

In [ ]:
%run _bootstrap.py

In [ ]:
from astra_tools.widgets.selectors import discover_sim_runs, PhaseStepper

runs = discover_sim_runs(SIM_DIR)
if not runs:
    from IPython.display import display, HTML
    display(HTML("<div style='background:#ffe0e0;border:1px solid #e57373;padding:8px 12px'>未找到 ASTRA 输出文件 — 请先运行 02_astra.ipynb 完成一次追踪</div>"))
    raise SystemExit("未找到 ASTRA 输出 (请先运行 02_astra)")
stem = sorted(runs)[0]
run_key = sorted(runs[stem])[-1]   # 最新一次 run (批 4: 不再硬编码 001)
phase_files = sorted(runs[stem][run_key][t] for t in runs[stem][run_key]
                     if t.lstrip("-").isdigit())
if not phase_files:
    from IPython.display import display, HTML
    display(HTML("<div style='background:#ffe0e0;border:1px solid #e57373;padding:8px 12px'>没有相空间输出 (OUTPUT 中 PhaseS=F?) — 请先运行 02_astra.ipynb</div>"))
    raise SystemExit("没有相空间输出 (OUTPUT 中 PhaseS=F?)")
print("发现相空间文件: %d 个 (stem=%s)" % (len(phase_files), stem))

# 步进器: 滑块 + ◀◀ ◀ ▶ ▶▶, 对应 postpro 的 z 位置步进功能
stepper = PhaseStepper(phase_files)
stepper

In [ ]:
# 统计表 + 相空间图: 随步进自动刷新
from pathlib import Path
from IPython.display import display
from ipywidgets import interactive_output
import ipywidgets as widgets
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics
from astra_tools.widgets.panels import distribution_summary_html, stats_table_html, display_bz_warning
from astra_tools.deck.solenoid import solenoid_bz_at_z
from IPython.display import display, HTML
from astra_tools.plot.phase_space import plot_transverse_phase_space, plot_phase_space

out = widgets.Output()

def _update(i):
    with out:
        out.clear_output(wait=True)
        dist = read_distribution(stepper.path)
        distribution_summary_html(dist)
        bz = solenoid_bz_at_z(SIM_DIR / "astra.in", dist.ref_z_m)
        if bz is None:
            display_bz_warning(SIM_DIR)
        else:
            display(HTML("<div style='background:#e8f5e9;border:1px solid #81c784;padding:6px 10px;margin:4px 0'>螺线管正则动量已自动应用: Bz(束心) = %.4f T (手册 4.13.1)</div>" % bz))
        stats_table_html(compute_statistics(dist, bz_on_axis_T=bz or 0.0))
        plot_transverse_phase_space(dist, bz_on_axis_T=bz or 0.0)
        # 高能束流 (1 GeV, εn=1 um) 的 x' 只有 ~1 urad, 原始图必然
        # 是一条贴地横线 (物理正确, 与 Xemit 一致); normalize=True
        # 除以各自 sigma 后结构清晰可见
        plot_phase_space(dist, plane="x", normalize=True,
                         title="x-x' normalized", bz_on_axis_T=bz or 0.0)
        plot_phase_space(dist, plane="z")

display(interactive_output(_update, {"i": stepper.index}))
_update(stepper.index.value)

# 下方单元 (6D/切割/切片/BFF/导出) 使用这个 dist; 步进到新 z 后
# 重跑本单元及需要更新的下方单元即可
dist = read_distribution(stepper.path)
print("当前文件:", stepper.path.name)

In [ ]:
# 任意参数对控件 (选择参数 + 加投影/减相关/状态着色/叠加)
from IPython.display import display
from astra_tools.widgets.selectors import PhaseSpaceParamSelector
from astra_tools.plot.arbitrary_phase_space import plot_arbitrary

param_sel = PhaseSpaceParamSelector(dist)
display(param_sel)
display(plot_arbitrary(dist, "x", "E_kin", subtract_corr=True,
                       title="x vs E_kin (linear correlation removed)"))


## 任意参数对相空间 (postpro 5.6.2 菜单 2)

任意两参数散点 (x/y/z/px/py/pz/clock/t/x'/y'/dp-p/E_kin) + 加投影、
减线性相关、状态着色、多分布叠加。


In [ ]:
# 核心发射度曲线 (当前束团, 手册 4.13.5) + 核心亮度演化 (Xemit/Yemit/LandF)
from IPython.display import display
from astra_tools.plot.advanced_plots import plot_core_emittance_curve, plot_core_brightness
from astra_tools.io import read_emit_files, parse_output_file

display(plot_core_emittance_curve(dist, title="core emittance vs particle fraction (4.13.5)"))

try:
    _rn = str(run_key)
    _emit = read_emit_files(str(SIM_DIR / stem), run=_rn)
    _landf = parse_output_file(str(SIM_DIR / (stem + ".LandF." + _rn)))
    _ce = dict(mean_z=_emit.x.z, norm_emit_x=_emit.x.emit,
               norm_emit_y=_emit.y.emit)
    display(plot_core_brightness(_ce, _landf,
                                 title="transverse core brightness (5.6.1 项 8)"))
except Exception as _e:
    print("核心亮度演化不可用 (需 Xemit/Yemit/LandF 输出):", _e)


## 核心发射度 (手册 4.13.5, 菜单 1 项 6) 与核心亮度 (项 8)

核心发射度: 按单粒子发射度不变量排序取核心, 100% 处 = 标准 rms 发射度
(f=1 与 ASTRA 精确一致; f<1 趋势一致, 数值与 ASTRA Cemit 有算法差异)。
核心亮度 B = Q/(εnx·εny) vs z (用 Xemit/Yemit/LandF 演化)。


In [ ]:
# 时间坐标: 纵向相空间(t) + 三视图 vs time + 状态着色 (手册 Table 6)
from IPython.display import display
from astra_tools.plot.phase_space import plot_phase_space
from astra_tools.plot.overview import plot_overview

display(plot_phase_space(dist, plane="t",
                         title="longitudinal phase space (time)"))
display(plot_overview(dist, time=True, title="bunch views vs time"))
display(plot_phase_space(dist, plane="x", color_by_status=True,
                         title="x-x' by particle status (Table 6)"))


## 时间坐标视图 (postpro 5.6.1 项 3/5)

t = (z-⟨z⟩)/(β̄c) 已发射粒子, 未发射 (status -1..-6) 用发射时间 clock。
状态着色按手册 Table 6 (secondary/normal/marked/passive/cathode/lost)。


**步进器用法**: 点击 ◀ / ▶ (或 ◀◀ / ▶▶, 或直接拖滑块) 切换 z 位置;
上方统计表与三张相空间图**自动刷新**, 下方 6D 全景/切割/切片/BFF/
导出等单元在步进后重跑即可 (与 postpro 的操作逻辑一致: 步进改变
"当前束团", 再选择要看的图)。

In [ ]:
# 6D 全景与投影
from astra_tools.plot.overview import plot_overview, plot_transverse_profile
from astra_tools.plot.distributions import plot_distributions, plot_energy_distribution
fig, _ = plot_overview(dist)
plot_transverse_profile(dist)
plot_distributions(dist)
plot_energy_distribution(dist)

In [ ]:
# z-plot (所有粒子沿束线的位置, 含丢失粒子)
from astra_tools.plot.advanced_plots import plot_z_plot
plot_z_plot(dist)

In [ ]:
# 相空间切割 (postpro 5.6.4): 修改窗口后重跑本单元
from astra_tools.analysis.cuts import cut_distribution
from astra_tools.widgets.panels import stats_table_html
from astra_tools.analysis.statistics import compute_statistics

dist_cut, mask = cut_distribution(dist, x_range=(-1e-3, 1e-3))
print("切割后 (x ±1 mm): 保留 %d/%d 粒子" % (dist_cut.n_active, dist.n_active))
stats_table_html(compute_statistics(dist_cut))

In [ ]:
# 优化切割 + 交互切割控件 + 保存新分布
from IPython.display import display
from astra_tools.analysis.cuts import optimized_cut
from astra_tools.widgets.selectors import CutControls
from astra_tools.io import write_distribution

d_opt, _mask = optimized_cut(dist, width=5e-3, param="z")
print("优化切割 (z 窗口 5 mm): 保留 %d/%d 粒子"
      % (d_opt.n_active, dist.n_active))

# 交互切割: 滑块调窗口, 应用/撤销
cut_ctl = CutControls(dist)
display(cut_ctl)

# 保存新分布供继续追踪 (写入 SIM_DIR/export, 二进制 ASTRA 格式)
_out = SIM_DIR / "export"
_out.mkdir(parents=True, exist_ok=True)
_p = write_distribution(dist, _out / "postpro_cut.ini")
print("已保存分布:", _p)


## 相空间操作 (postpro 5.6.4)

优化切割 (interval 参数 → 存活粒子数最大) + 交互切割控件
(滑块 + 应用/撤销) + 保存新分布供继续追踪。


In [ ]:
# 3D slice 椭圆与失配参数
from astra_tools.plot.advanced_plots import plot_slice_ellipses_3d, plot_slice_mismatch
plot_slice_ellipses_3d(dist, n_slices=10)
plot_slice_mismatch(dist, n_slices=10)

## 纵向切片 / BFF / 数据导出 (原 07 高级分析)

In [ ]:
from astra_tools.analysis.slices import compute_slice_analysis
from astra_tools.plot.slice_plots import plot_slice_dashboard
sa = compute_slice_analysis(dist, n_slices=20)
plot_slice_dashboard(sa)

In [ ]:
# slice 增强: 按能量切片 + 发散角 z-投影 + 投影切换 + 减相关 + 改关联能散
from IPython.display import display
from astra_tools.analysis.slices import compute_slice_analysis
from astra_tools.plot.slice_plots import plot_slice_sizes, plot_slice_dashboard
from astra_tools.plot.advanced_plots import plot_slice_ellipses_3d
from astra_tools.analysis.cuts import modify_correlated_energy_spread

# 项 8/9 按能量切片 + 项 5 发散角 z-投影
sa_e = compute_slice_analysis(dist, n_slices=20, binning="equi_energy")
display(plot_slice_sizes(sa_e, divergences=True,
                         title="slice sizes + divergences (w.r.t. energy)"))

# 项 7 投影切换 (y-y') + 项 11 减线性相关
display(plot_slice_ellipses_3d(dist, n_slices=8, plane="yyp",
                               title="3D slice ellipses (y-y', plane switch)"))
display(plot_slice_ellipses_3d(dist, n_slices=8, subtract_corr=True,
                               title="3D slice ellipses (lin. corr removed)"))

# 项 12 改关联能散 (factor=0 去相关后重算 slice)
d_corr = modify_correlated_energy_spread(dist, factor=0.0)
display(plot_slice_dashboard(compute_slice_analysis(d_corr, n_slices=20),
                             title="slice dashboard (correlated E spread removed)"))


## slice 增强 (postpro 5.6.3 项 5/7/8/9/11/12)

按能量切片 (项 8/9)、发散角 z-投影 (项 5)、投影切换 (项 7)、
减线性相关 (项 11)、改关联能散 (项 12)。


In [ ]:
from astra_tools.analysis.bff import compute_bff
from astra_tools.plot.bff_plots import plot_bff_with_amplitude
bff = compute_bff(dist.filter_active().z, dist.filter_active().charge,
                  kmin=10, kmax=1e5, nk=150, detect_features=True)
plot_bff_with_amplitude(bff)

In [ ]:
# slice 失配参数 (zeta >= 1, 匹配越好越接近 1)
from astra_tools.plot.advanced_plots import plot_slice_mismatch
plot_slice_mismatch(dist, n_slices=20)

In [ ]:
from astra_tools.export import export_distribution, export_statistics
from astra_tools.analysis.statistics import compute_statistics
out = SIM_DIR / "export"
print("导出目录:", out)
print(export_distribution(dist, out))
print(export_statistics(compute_statistics(dist), out))

## 束包络 + 孔径几何叠加 (Aperture 算例)

In [ ]:
from astra_tools.plot.advanced_plots import (
    aperture_elements, plot_envelope_with_aperture)
from astra_tools.namelist.parse import parse_namelists
from astra_tools.io.astra_emit import read_emit_files

ap = parse_namelists(PROJECT_ROOT / "examples/Aperture/astra.in")["APERTURE"]
emit_ap = read_emit_files(str(PROJECT_ROOT / "examples/Aperture/golden/astra"))
plot_envelope_with_aperture(emit_ap, aperture_elements(ap))

## 核心电荷分数曲线 (核心束长/发射度 vs 电荷分数)

In [ ]:
from astra_tools.plot.advanced_plots import plot_core_fraction_curves
plot_core_fraction_curves(dist)

## 投影 slice 椭圆 (5.6.3 项 4/7/11, 2D)

投影 rms slice 发射度椭圆 (2D), 投影切换 (x-x' / y-y' / x-y' / y-x')
与减线性相关 (项 11)。


In [ ]:
# 投影 rms slice 椭圆 (2D) + 投影切换 + 减线性相关
from astra_tools.plot.advanced_plots import plot_slice_ellipses_2d
plot_slice_ellipses_2d(dist, n_slices=10, plane="xxp")
plot_slice_ellipses_2d(dist, n_slices=10, plane="yyp")
plot_slice_ellipses_2d(dist, n_slices=10, plane="xxp", subtract_corr=True)


## Plot_steering.par: 混合分布按粒子索引着色 (手册 5.6, Plot_mode=1)

CP_ind_1..15 将粒子索引映射到 RGB; 黑色 (0,0,0) = 不绘制。
此处用默认色替代黑色并提示, 相空间图按粒子索引着色。


In [ ]:
# Plot_steering.par: 混合分布按粒子索引着色 (CP_ind, 手册 5.6)
from astra_tools.io.plot_steering import read_plot_steering, cp_index_colors
from astra_tools.plot.phase_space import plot_phase_space
from pathlib import Path
import numpy as np
psf = SIM_DIR / "Plot_steering.par"
if psf.exists():
    ps = read_plot_steering(psf)
    if ps.get("cp_ind"):
        cols = cp_index_colors(dist.index, ps["cp_ind"])
        n_black = int(np.sum([c is None for c in cols]))
        # 黑色 (0,0,0) = 不绘制; 这里用默认色替代并提示
        cols = np.array([c if c is not None else "#0077BB" for c in cols],
                        dtype=object)
        plot_phase_space(dist, plane="x", colors=cols,
                         title="x-x' colored by particle index (CP_ind)")
        if n_black:
            print("%d 个黑色 (不绘制) 粒子已用默认色替代" % n_black)
    else:
        print("Plot_steering.par 无 CP_ind 定义 (混合分布索引着色未启用)")
else:
    print("无 Plot_steering.par (Steering_parameters 重定向/着色未启用)")
